## Connect to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

# Corrected path to your CSV file in Google Drive
csv_file_path = '/content/drive/MyDrive/Asset Mapping Project/Final content/Processed_DFW_Assets_Master.csv' # @param {type:"string"}
df = pd.read_csv(csv_file_path)
display(df.head())

,name,building,county,Nearest_Street,latitude,longitude,Building_Sq_Ft,Roof_Sq_Ft,Roof_Sq_M,Annual_kWh,...,Daily_kWh,Households_Powered,Households_1mi,Households_3mi,EV_Stations_1mi,EV_Stations_3mi,Priority_Score,Annual_Class_BP,Household_Class_BP,EV_Class_BP
0,NorthPark Center,retail,"Dallas County, Texas",North Central Expressway,32.868393,-96.773500,1.219857e+06,975885.37,90659.750873,29119912.0,...,79780.6,2214.1,11474,67244,4,14,0.3172,High,High,Medium
1,Industrial on Lance Drive,industrial,"Dallas County, Texas",Lance Drive,32.717254,-97.030512,1.206589e+06,965270.89,89673.665681,28803181.0,...,78912.8,2190.0,2673,42009,1,18,0.1844,High,Low,Low
2,Commercial on North Central Expressway,commercial,"Dallas County, Texas",North Central Expressway,32.979636,-96.715736,5.527943e+04,44223.54,4108.366866,1319607.0,...,3615.4,100.3,4517,43514,11,45,0.0740,Medium,Medium,Medium
3,Unbridled Living of Dallas,commercial,"Dallas County, Texas",Coit Road,32.916452,-96.770310,1.151029e+05,92082.31,8554.446599,2747688.0,...,7527.9,208.9,6819,66022,5,27,0.1154,Medium,Medium,Medium
4,Office on Hillcrest Road,office,"Dallas County, Texas",Hillcrest Road,32.923485,-96.785250,5.662391e+04,45299.13,4208.289177,1351702.0,...,3703.3,102.8,4719,65889,7,40,0.0771,Medium,Medium,Medium


In [7]:
# Convert the DataFrame to a GeoDataFrame
gdf_final = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326"
)

In [18]:
# Load EV station data from CSV and convert to GeoDataFrame
ev_stations_csv_path = '/content/drive/MyDrive/Asset Mapping Project/Final content/EV_Stations_DFW (2).csv' # @param {type:"string"}
ev_df = pd.read_csv(ev_stations_csv_path)

# Assuming 'longitude' and 'latitude' columns exist in your EV data
ev_gdf_proj = gpd.GeoDataFrame(
    ev_df, geometry=gpd.points_from_xy(ev_df.longitude, ev_df.latitude), crs="EPSG:4326"
)
display(ev_gdf_proj.head())

,id,station_name,street_address,city,state,zip,latitude,longitude,ev_level1_evse_num,ev_level2_evse_num,ev_dc_fast_num,total_chargers,geometry
0,39714,Grubbs Nissan,310d Airport Fwy,Bedford,TX,76022,32.839448,-97.160732,0.0,1.0,2.0,3.0,POINT (-97.16073 32.83945)
1,39723,Nissan - Fort Worth,3451 W Loop 820 S,Fort Worth,TX,76116,32.721017,-97.478446,0.0,1.0,1.0,2.0,POINT (-97.47845 32.72102)
2,39742,Trophy Nissan,5031 N Galloway Ave,Mesquite,TX,75150,32.829506,-96.629552,0.0,2.0,1.0,3.0,POINT (-96.62955 32.82951)
3,39761,Southwest Nissan,3050 Fort Worth Hwy,Weatherford,TX,76087,32.756100,-97.705239,0.0,1.0,0.0,1.0,POINT (-97.70524 32.7561)
4,47511,City of Dallas,10011 Log Cabin Rd,Dallas,TX,75253,32.653718,-96.628439,0.0,1.0,0.0,1.0,POINT (-96.62844 32.65372)


In [21]:
import folium
from folium import plugins
import json
import geopandas as gpd
import pandas as pd # Ensure pandas is imported as it's used within the cell

# Define CRS constants
CRS_GEOGRAPHIC = "EPSG:4326"  # WGS84 geographic coordinate system
# A common projected CRS for North Central Texas (example, may need adjustment)
CRS_PROJECTED = "EPSG:2276" # NAD83 / Texas North Central (ftUS)

# Define OUTPUT_DIR
OUTPUT_DIR = '/content/outputs' # Default output directory
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Work with WGS84 for Folium (Leaflet uses lat/lon)
gdf_map = gdf_final.copy()

# --- Create DFW Metroplex outline and County outlines in WGS84 ---
# tracts_gdf is in CRS_PROJECTED, reproject to WGS84 for Folium
# NOTE: `tracts_gdf` is not defined in this notebook state. This section will cause an error if uncommented.
# For now, I'm commenting it out based on previous interaction context.
# tracts_gdf_wgs84 = tracts_gdf.to_crs(CRS_GEOGRAPHIC)

# # Dissolve all DFW tracts to form a single Metroplex boundary
# dfw_metroplex_boundary_gdf = tracts_gdf_wgs84.dissolve()
# dfw_metroplex_geometry = dfw_metroplex_boundary_gdf.geometry.iloc[0]

# # Dissolve tracts by county to get individual county boundaries
# dfw_county_boundaries_gdf = tracts_gdf_wgs84.dissolve(by='county_name').reset_index()

# --- Color scheme by building type ---
BUILDING_COLORS = {
    'commercial'  : '#4A90D9',   # blue
    'warehouse'   : '#E67E22',   # orange
    'industrial'  : '#8E44AD',   # purple
    'retail'      : '#27AE60',   # green
    'office'      : '#2C3E50',   # dark navy
    'supermarket' : '#E74C3C',   # red
    'other'       : '#95A5A6',   # gray for unknown/other types
}

def get_color(building_type):
    return BUILDING_COLORS.get(str(building_type).lower(), BUILDING_COLORS['other'])

# --- DFW center coordinates ---
DFW_CENTER = [32.89, -97.04]

# --- Build the Folium map ---
m = folium.Map(
    location=DFW_CENTER,
    zoom_start=10,
    tiles='CartoDB positron',
    control_scale=True
)

# Add additional tile layer options
folium.TileLayer('CartoDB dark_matter', name='Dark basemap').add_to(m)
folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)

# --- Feature groups (layers) ---
# Create a FeatureGroup for each building type for interactive toggling
# Set control=False to prevent them from appearing in the default LayerControl
building_type_layers = {}
for btype, color in BUILDING_COLORS.items():
    # Name the FeatureGroup explicitly, so we can reference it in JS
    # Set show=False so that custom JS fully controls initial visibility
    fg = folium.FeatureGroup(name=f"Building Type: {btype.title()}", show=False, control=False)
    building_type_layers[btype] = fg
    fg.add_to(m) # Add to map immediately

# Generic layer for EV stations
layer_ev_stations  = folium.FeatureGroup(name='EV charging stations', show=False)
layer_ev_stations.add_to(m) # Add to map

# Placeholder layers for DFW outline and county outlines (if uncommented later)
layer_dfw_outline  = folium.FeatureGroup(name='DFW Metroplex Outline', show=False)
layer_dfw_outline.add_to(m)
layer_county_outlines = folium.FeatureGroup(name='DFW County Outlines', show=False)
layer_county_outlines.add_to(m)


# --- Add EV stations ---
# This section is now uncommented and assumes `ev_gdf_proj` exists from the previous step
if 'ev_gdf_proj' in locals() and isinstance(ev_gdf_proj, gpd.GeoDataFrame) and not ev_gdf_proj.empty:
    ev_wgs = ev_gdf_proj.to_crs(CRS_GEOGRAPHIC)
    for _, ev_row in ev_wgs.iterrows():
        if pd.notna(ev_row.geometry.y):
            folium.CircleMarker(
                location=[ev_row.geometry.y, ev_row.geometry.x],
                radius=4,
                color='#2ECC71',
                fill=True,
                fill_color='#2ECC71',
                fill_opacity=0.8,
                tooltip=str(ev_row.get('station_name', 'EV Station')),
                popup=folium.Popup(f"""
                    <b>{ev_row.get('station_name','EV Station')}</b><br>
                    {ev_row.get('street_address','')}
                """, max_width=250)
            ).add_to(layer_ev_stations)

# --- Add DFW Metroplex outline to its layer ---
# This section is commented out because `dfw_metroplex_geometry` is not defined in current notebook state.
# if not dfw_metroplex_geometry.is_empty:
#     folium.GeoJson(
#         dfw_metroplex_geometry,
#         name='DFW Metroplex Outline',
#         style_function=lambda x: {
#             'fillColor': 'none',
#             'color': 'black',
#             'weight': 3,
#             'dashArray': '5, 5'
#         }
#     ).add_to(layer_dfw_outline)

# --- Add DFW county outlines to its layer ---
# This section is commented out because `dfw_county_boundaries_gdf` is not defined in current notebook state.
# folium.GeoJson(
#     dfw_county_boundaries_gdf.to_json(),
#     name='DFW County Outlines',
#     style_function=lambda x: {
#         'fillColor': 'none',
#         'color': 'grey',
#         'weight': 1,
#         'dashArray': '3, 3'
#     },
#     tooltip=folium.features.GeoJsonTooltip(fields=['county_name'])
# ).add_to(layer_county_outlines)

# --- Add building markers with rich popups ---
buildings_js_data = [] # Retaining for potential multi-select feature

for idx, row in gdf_map.iterrows():
    lat = row['latitude']
    lon = row['longitude']

    if pd.isna(lat) or pd.isna(lon):
        continue

    building_type_key = str(row.get('building', 'other')).lower()
    color = get_color(building_type_key)

    # Build a rich HTML popup
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; min-width: 280px;">
        <h4 style="margin:0 0 8px 0; color:#2C3E50;">{row.get('name','Unknown')}</h4>
        <hr style="margin:4px 0;">
        <table style="width:100%; border-collapse:collapse;">
            <tr><td><b>Type:</b></td><td>{row.get('building','—')}</td></tr>
            <tr><td><b>County:</b></td><td>{str(row.get('county','')).replace(', Texas','')}</td></tr>
            <tr><td><b>Street:</b></td><td>{row.get('Nearest_Street','—')}</td></tr>
            <tr><td><b>Building Sq Ft:</b></td><td>{row.get('Building_Sq_Ft',0):,.0f}</td></tr>
            <tr><td><b>Roof Sq Ft:</b></td><td>{row.get('Roof_Sq_Ft',0):,.0f}</td></tr>
        </table>
        <hr style="margin:6px 0;">
        <b>Solar Energy Potential</b>
        <table style="width:100%; border-collapse:collapse; margin-top:4px;">
            <tr><td>Annual generation:</td><td><b>{row.get('Annual_MWh',0):,.1f} MWh/yr</b></td></tr>
            <tr><td>Households powered:</td><td><b>{row.get('Households_Powered',0):,.0f}</b></td></tr>
        </table>
        <hr style="margin:6px 0;">
        <b>1-Mile Buffer</b>
        <table style="width:100%; border-collapse:collapse; margin-top:4px;">
            <tr><td>Households:</td><td>{row.get('Households_1mi',0):,}</td></tr>
            <tr><td>EV stations:</td><td>{row.get('EV_Stations_1mi',0)}</td></tr>
        </table>
        <b>3-Mile Buffer</b>
        <table style="width:100%; border-collapse:collapse; margin-top:4px;">
            <tr><td>Households:</td><td>{row.get('Households_3mi',0):,}</td></tr>
            <tr><td>EV stations:</td><td>{row.get('EV_Stations_3mi',0)}</td></tr>
        </table>
        <hr style="margin:6px 0;">
        <span style="color:#888; font-size:11px;">Priority Score: {row.get('Priority_Score',0):.4f}</span>
        <br>
        <button onclick="drawBuffers('{m.get_name()}', {lat},{lon},'1mile')"
            style="margin-top:6px; padding:4px 8px; background:#4A90D9; color:white; border:none; border-radius:4px; cursor:pointer;">
            Show 1-mi buffer
        </button>
        <button onclick="drawBuffers('{m.get_name()}', {lat},{lon},'3mile')"
            style="margin-top:4px; padding:4px 8px; background:#E67E22; color:white; border:none; border-radius:4px; cursor:pointer;">
            Show 3-mi buffer
        </button>
    </div>
    """

    # Add marker to its specific building type layer
    fg_to_add = building_type_layers.get(building_type_key, building_type_layers['other'])
    folium.CircleMarker(
        location=[lat, lon],
        radius=max(4, min(12, row.get('Annual_MWh', 0) / 500)),  # size by energy potential
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        weight=0.8,
        tooltip=f"{row.get('name','Building')} — {row.get('Annual_MWh',0):,.0f} MWh/yr",
        popup=folium.Popup(popup_html, max_width=320)
    ).add_to(fg_to_add)

    # Store data for JavaScript multi-select
    buildings_js_data.append({
        'lat'         : round(lat, 6),
        'lon'         : round(lon, 6),
        'name'        : str(row.get('name','')),
        'building'    : str(row.get('building','')),
        'annual_mwh'  : float(row.get('Annual_MWh', 0)),
        'hh_powered'  : float(row.get('Households_Powered', 0)),
        'hh_1mi'      : int(row.get('Households_1mi', 0)),
        'hh_3mi'      : int(row.get('Households_3mi', 0)),
        'ev_1mi'      : int(row.get('EV_Stations_1mi', 0)),
        'ev_3mi'      : int(row.get('EV_Stations_3mi', 0)),
    })


# --- Custom Interactive Building Type Legend ---
custom_legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px 16px; border-radius:8px;
     border:1px solid #ccc; font-family:Arial; font-size:12px; box-shadow:2px 2px 6px rgba(0,0,0,0.2);">
    <b>Building Types</b><br>
"""

for btype, color in BUILDING_COLORS.items():
    # Use the name assigned to the FeatureGroup for lookup in JavaScript
    fg_name = f"Building Type: {btype.title()}"
    custom_legend_html += f"""
    <label style="display:block; margin-bottom:4px;">
        <input type="checkbox" class="building-type-checkbox" data-layer-name="{fg_name}">
        <span style="background-color:{color}; display:inline-block; width:10px; height:10px; border-radius:50%; margin-right:5px;"></span>
        {btype.title()}
    </label>
    """

custom_legend_html += """
    <hr style="margin:6px 0;">
    <i>Marker size = energy potential</i>
</div>
"""
m.get_root().html.add_child(folium.Element(custom_legend_html))


# --- Folium's default LayerControl for base maps and EV stations ---
# This will control the base map layers and EV stations, but not the building types
# (as building types are controlled by the custom legend).
folium.LayerControl(collapsed=False).add_to(m)


# --- JavaScript for dynamic buffer drawing and interactive legend ---
js_code = f"""
<script>
var activeBuffers = [];

function drawBuffers(mapName, lat, lon, bufferType) {{
    var mapObj = window[mapName];
    if (!mapObj) {{ console.error("Map object '", mapName, "' not found."); return; }}
    activeBuffers.forEach(function(layer) {{ mapObj.removeLayer(layer); }});
    activeBuffers = [];

    var radiusMeters = bufferType === '1mile' ? 1609.34 : 4828.03;
    var color = bufferType === '1mile' ? '#4A90D9' : '#E67E22';
    var label = bufferType === '1mile' ? '1-mile buffer' : '3-mile buffer';

    var circle = L.circle([lat, lon], {{
        radius: radiusMeters,
        color: color,
        fillColor: color,
        fillOpacity: 0.12,
        weight: 2,
        dashArray: '6 4'
    }}).addTo(mapObj);
    circle.bindTooltip(label, {{permanent: false}});
    activeBuffers.push(circle);
    mapObj.fitBounds(circle.getBounds());
}}

// ---- Custom Layer Toggle for Building Types ----
document.addEventListener('DOMContentLoaded', function() {{
    var mapObj = window['{m.get_name()}'];
    if (!mapObj) {{
        console.error("Map object '{m.get_name()}' not found for legend interaction.");
        return;
    }}

    // Handle initial state: add layers if checkboxes are checked (if they were `checked` by default) or let them be hidden
    document.querySelectorAll('.building-type-checkbox').forEach(function(checkbox) {{
        var layerName = checkbox.dataset.layerName;
        var layerFound = false;

        mapObj.eachLayer(function(layer) {{
            if (layer.options && layer.options.name === layerName) {{
                if (checkbox.checked) {{
                    mapObj.addLayer(layer);
                }} else {{
                    mapObj.removeLayer(layer); // Ensure it's removed if initially unchecked
                }}
                layerFound = true;
            }}
        }});
        if (!layerFound) {{
            console.warn('Layer with name ' + layerName + ' not found during initial check.');
        }}

        checkbox.addEventListener('change', function() {{
            var layerName = this.dataset.layerName;
            var layerFound = false;

            // Iterate through Leaflet layers to find the FeatureGroup by its assigned 'name' option
            mapObj.eachLayer(function(layer) {{
                // Check if the layer is a FeatureGroup (has options.name) and its name matches
                if (layer.options && layer.options.name === layerName) {{
                    if (checkbox.checked) {{
                        mapObj.addLayer(layer);
                    }} else {{
                        mapObj.removeLayer(layer);
                    }}
                    layerFound = true;
                }}
            }});

            if (!layerFound) {{
                console.warn('Layer with name ' + layerName + ' not found.');
            }}
        }});
    }});

    // ---- Multi-select feature (from original code, kept for consistency) ----
    var selectedBuildings = [];
    var allBuildingData = {json.dumps(buildings_js_data[:500])};

    function addToSelection(lat, lon, name, mwh, hh1, hh3, ev1, ev3) {{
        selectedBuildings.push({{lat:lat, lon:lon, name:name, mwh:mwh, hh1:hh1, hh3:hh3, ev1:ev1, ev3:ev3}});
        document.getElementById('sel-count').innerText = selectedBuildings.length;
        document.getElementById('multi-panel').style.display = 'block';
    }}

    function analyzeSelection() {{
        if (selectedBuildings.length === 0) return;

        var centLat = selectedBuildings.reduce((s,b) => s+b.lat, 0) / selectedBuildings.length;
        var centLon = selectedBuildings.reduce((s,b) => s+b.lon, 0) / selectedBuildings.length;

        var totalMwh = selectedBuildings.reduce((s,b) => s+b.mwh, 0);
        var totalHH1  = selectedBuildings.reduce((s,b) => s+b.hh1, 0);
        var totalHH3  = selectedBuildings.reduce((s,b) => s+b.hh3, 0);
        var totalEV1  = selectedBuildings.reduce((s,b) => s+b.ev1, 0);

        var mapObj = window['{m.get_name()}'];
        if (!mapObj) {{
            console.error("Map object '", '{m.get_name()}', "' not found for analysis.");
            return;
        }}

        activeBuffers.forEach(function(l) {{ mapObj.removeLayer(l); }});
        activeBuffers = [];

        var buf1 = L.circle([centLat, centLon], {{
            radius: 1609.34, color:'#4A90D9', fillColor:'#4A90D9',
            fillOpacity: 0.10, weight:2, dashArray:'6 4'
        }}).addTo(mapObj).bindTooltip('1-mi centroid buffer');

        var buf3 = L.circle([centLat, centLon], {{
            radius: 4828.03, color:'#E67E22', fillColor:'#E67E22',
            fillOpacity: 0.07, weight:2, dashArray:'6 4'
        }}).addTo(mapObj).bindTooltip('3-mi centroid buffer');

        activeBuffers.push(buf1, buf3);

        var centMark = L.marker([centLat, centLon]).addTo(mapObj)
            .bindPopup('<b>Selection centroid</b><br>'
                + selectedBuildings.length + ' buildings selected<br>'
                + 'Total generation: <b>' + totalMwh.toFixed(1) + ' MWh/yr</b><br>'
                + 'Households in 1-mi buffer: <b>' + totalHH1.toLocaleString() + '</b><br>'
                + 'Households in 3-mi buffer: <b>' + totalHH3.toLocaleString() + '</b><br>'
                + 'EV stations in 1-mi buffer: <b>' + totalEV1 + '</b>'
            ).openPopup();
        activeBuffers.push(centMark);
        mapObj.setView([centLat, centLon], 12);
    }}

    function clearSelection() {{
        selectedBuildings = [];
        document.getElementById('sel-count').innerText = '0';
        document.getElementById('multi-panel').style.display = 'none';

        var mapObj = window['{m.get_name()}'];
        if (mapObj) {{
            activeBuffers.forEach(function(l) {{ mapObj.removeLayer(l); }});
        }}
        activeBuffers = [];
    }}

    document.addEventListener('click', function(e) {{
        if (e.target.classList.contains('leaflet-container')) {{
            clearSelection();
        }}
    }});

}});

</script>

<div id="multi-panel" style="display:none; position:fixed; top:80px; right:10px; z-index:2000;
     background:white; padding:12px; border-radius:8px; border:1px solid #ccc;
     font-family:Arial; font-size:12px; max-width:200px; box-shadow:2px 2px 6px rgba(0,0,0,0.2);">
    <b>Multi-select mode</b><br>
    <span id="sel-count">0</span> buildings selected<br>
    <button onclick="analyzeSelection()"
        style="margin-top:8px; width:100%; padding:5px; background:#27AE60; color:white; border:none; border-radius:4px; cursor:pointer;">
        Analyze selection
    </button>
    <button onclick="clearSelection()"
        style="margin-top:4px; width:100%; padding:5px; background:#E74C3C; color:white; border:none; border-radius:4px; cursor:pointer;">
        Clear
    </button>
</div>
"""
m.get_root().html.add_child(folium.Element(js_code))

# Save map
map_path = f"{OUTPUT_DIR}/DFW_Energy_Map.html"
m.save(map_path)
print(f"Interactive map saved: {map_path}")
print("Download this file and open it in any browser to use the map.")

Interactive map saved: /content/outputs/DFW_Energy_Map.html
Download this file and open it in any browser to use the map.
